In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py audio/*.py inference/models/*.py
!cd /kaggle/working/Real-ESRGAN && python inference.py --help >/dev/null
!cd /kaggle/working/Real-ESRGAN && python -m audio.process --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep av1_nvenc || true


In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

START_TIME = 5 * 60 + 35
TEST_SECONDS = 15


In [ ]:
# 视频参数
VIDEO_ENHANCE = True

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2

# RIFE_FPS = 0 关闭 RIFE 并保持源帧率；>0 时必须 >= 输入视频源帧率。
RIFE_FPS = 60

# GPU：False=单 GPU(cuda:0)，True=双 GPU(cuda:0,1)。
DUAL_GPU = True

# BasicVSR++ 固定参数。
BVS_TILE_SIZE = 640
BVS_CLIP_LENGTH = 13
BVS_BATCH_SIZE = 1
BVS_STRENGTH = 1.0

# ==============================
# RTX 4090 AV1 NVENC 收藏级配置
# 4K 60fps / 10-bit / HDR 色彩信令
# ==============================

VIDEO_CODEC = "av1_nvenc"
ENCODE_GPU = 0

# 质量预设
PRESET = "P7"

# 调优模式
TUNE = "HQ"

# AV1 只有 Main profile；10-bit 由 P010LE / highbitdepth 实现，不存在 HEVC 式 MAIN10。
PROFILE = "MAIN"
PIX_FMT = "P010LE"

# 码率控制
RC = "VBR"
CQ = 18
BITRATE = "0"

# 多遍分析
MULTIPASS = "FULLRES"

# 前瞻分析
# NVENC SDK 13.1：lookaheadDepth <= 31 - B帧数；B_FRAMES=3 时最大为 28。
B_FRAMES = 3
RC_LOOKAHEAD = 28

# 自适应量化
SPATIAL_AQ = 1
TEMPORAL_AQ = 1
AQ_STRENGTH = 8

# B帧参考
B_REF_MODE = "MIDDLE"

# GOP：4K60 下 240 帧 = 4 秒
GOP_SIZE = 240

# ==============================
# HDR10 色彩信令
# 注意：只写 BT.2020/PQ 色彩标记，不会把 SDR 内容自动 tone-map 成 HDR。
# ==============================
COLOR_PRIMARIES = "BT2020"
COLOR_TRC = "SMPTE2084"
COLORSPACE = "BT2020NC"


In [ ]:
# 音频参数
AUDIO_ENHANCE = True
AUDIO_CODEC = "aac"  # 增强开启时使用；关闭增强时自动 stream copy
AUDIO_BITRATE = "256k"


In [ ]:
import subprocess
import sys

effective_audio_codec = AUDIO_CODEC if AUDIO_ENHANCE else "copy"

if VIDEO_ENHANCE:
    command = [
        sys.executable, "/kaggle/working/Real-ESRGAN/inference.py",
        "--input", INPUT_VIDEO,
        "--output", OUTPUT_VIDEO,
        "--model", MODEL,
        "--model-path", MODEL_PATH,
        "--scale", str(SCALE),
        "--rife-fps", str(RIFE_FPS),
        "--gpu-ids", "0,1" if DUAL_GPU else "0",
        "--bvs-tile-size", str(BVS_TILE_SIZE),
        "--bvs-clip-length", str(BVS_CLIP_LENGTH),
        "--bvs-batch-size", str(BVS_BATCH_SIZE),
        "--bvs-strength", str(BVS_STRENGTH),
        "--video-codec", VIDEO_CODEC,
        "--cq", str(CQ),
        "--nvenc-preset", PRESET.lower(),
        "--encode-gpu", str(ENCODE_GPU),
        "--av1-profile", PROFILE.lower(),
        "--av1-pix-fmt", PIX_FMT.lower(),
        "--av1-tune", TUNE.lower(),
        "--av1-rc", RC.lower(),
        "--av1-bitrate", BITRATE,
        "--av1-multipass", MULTIPASS.lower(),
        "--av1-rc-lookahead", str(RC_LOOKAHEAD),
        "--av1-spatial-aq", str(SPATIAL_AQ),
        "--av1-temporal-aq", str(TEMPORAL_AQ),
        "--av1-aq-strength", str(AQ_STRENGTH),
        "--av1-b-ref-mode", B_REF_MODE.lower(),
        "--av1-b-frames", str(B_FRAMES),
        "--av1-gop-size", str(GOP_SIZE),
        "--av1-color-primaries", COLOR_PRIMARIES.lower(),
        "--av1-color-trc", COLOR_TRC.lower(),
        "--av1-colorspace", COLORSPACE.lower(),
        "--audio-codec", effective_audio_codec,
        "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME),
        "--test-seconds", str(TEST_SECONDS),
        "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe",
    ]
else:
    command = [
        sys.executable, "-m", "audio.process",
        "--input", INPUT_VIDEO,
        "--output", OUTPUT_VIDEO,
        "--audio-codec", effective_audio_codec,
        "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME),
        "--test-seconds", str(TEST_SECONDS),
        "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe",
    ]

if AUDIO_ENHANCE:
    command.append("--audio-enhance")

cwd = "/kaggle/working/Real-ESRGAN" if not VIDEO_ENHANCE else None
process = subprocess.Popen(
    command,
    cwd=cwd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
